# Haplotype data access — `Ag3`

This notebook covers the `Ag3` methods for accessing *phased* SNP data (haplotypes): which phasing analyses are available, the phased SNP calls themselves, and the raw haplotype site definitions. Phasing resolves which alleles at different heterozygous sites sit together on the same physical chromosome copy, producing two separate haplotypes per diploid sample instead of one unordered pair of calls per site.

In [1]:
import malariagen_data
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../../gcs_cache"),
    results_cache="../../results_cache",
)
ag3

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen2/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


<MalariaGEN Ag3 API client>
Storage URL                           : simplecache::gs://vo_agam_release_master_us_central1
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : /Users/katie.barr/malariagen-data-python/results_cache
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.8.0.post13+b769b728
Client location                       : England, United Kingdom
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.8.0.post13+b769b728/Ag3.html

## `phasing_analysis_ids`

A read-only property (no parameters) returning the identifiers of the haplotype phasing analyses available for this data resource. These are the valid values for the `analysis` parameter accepted by `haplotypes` and `haplotype_sites`. For *Ag3*, phasing was run separately for different taxon groupings: `gamb_colu_arab` (all three taxa phased together), `gamb_colu` (gambiae + coluzzii only), and `arab` (arabiensis only) — mirroring the site-mask groupings seen in `site_mask_ids`.

In [2]:
ag3.phasing_analysis_ids

('gamb_colu_arab', 'gamb_colu', 'arab')

## `haplotypes`

Access phased SNP genotype calls for a genome region and set of samples, returned as an `xarray.Dataset` with dimensions `variants` (number of phased SNP sites), `alleles` (2 — reference and alternate), `samples`, and `ploidy` (2). Unlike unphased `call_genotype` from `snp_calls`, here the two values along the `ploidy` axis represent the allele on haplotype 1 and haplotype 2 respectively, consistently ordered across all sites for a given sample — i.e. you can slice out `call_genotype[..., 0]` and `call_genotype[..., 1]` as two independent haploid sequences per sample. Parameters:

- **region**: genome region(s) to query (contig, region string, gene ID, or a list of these).
- **analysis**: which phasing analysis to use (see `phasing_analysis_ids`); `"default"` uses the resource's configured default.
- **sample_sets**: which sample set(s)/release(s) to include. Note that not every sample set necessarily has phased data for every analysis — a `ValueError` is raised if none of the requested sample sets have data for the chosen `analysis`.
- **sample_query**: pandas query string to select samples from the metadata, e.g. by `taxon` or `country`.
- **sample_query_options**: extra kwargs passed through to pandas `query()`/`eval()`.
- **inline_array**, **chunks**: dask performance/chunking knobs, as for the SNP-data methods.
- **cohort_size**, **min_cohort_size**, **max_cohort_size**, **random_seed**: cohort down-sampling controls, same semantics as `snp_calls`.

**Diagram opportunity:** a small illustration of one heterozygous diploid genotype (e.g. `0/1`) being resolved by phasing into two ordered haplotype alleles (`0` on haplotype 1, `1` on haplotype 2), repeated across a few adjacent sites, to show visually what "phased" adds over the raw `call_genotype` from `snp_calls`. Would fit well right here, before the code example.

The example accesses phased calls for the `gamb_colu` analysis (gambiae + coluzzii) across a 1 Mbp region, restricted to *coluzzii* samples from one sample set.

In [3]:
ds_hap = ag3.haplotypes(
    region="3L:15,000,000-16,000,000",
    analysis="gamb_colu",
    sample_sets="AG1000G-BF-A",
    sample_query="taxon == 'coluzzii'",
)
ds_hap

Access haplotypes: ⠋ (0:00:00.00)

Access haplotypes: ⠙ (0:00:00.08)

Access haplotypes: ⠹ (0:00:00.17)

Load sample metadata: ⠋ (0:00:00.00)

<xarray.Dataset> Size: 44MB
Dimensions:           (variants: 258679, alleles: 2, samples: 82, ploidy: 2)
Coordinates:
    variant_position  (variants) int32 1MB dask.array<chunksize=(18291,), meta=np.ndarray>
    variant_contig    (variants) uint8 259kB dask.array<chunksize=(258679,), meta=np.ndarray>
    sample_id         (samples) object 656B dask.array<chunksize=(82,), meta=np.ndarray>
Dimensions without coordinates: variants, alleles, samples, ploidy
Data variables:
    variant_allele    (variants, alleles) |S1 517kB dask.array<chunksize=(18291, 1), meta=np.ndarray>
    call_genotype     (variants, samples, ploidy) int8 42MB dask.array<chunksize=(18291, 60, 2), meta=np.ndarray>
Attributes:
    contigs:   ('2R', '2L', '3R', '3L', 'X')
    analysis:  gamb_colu

## `haplotype_sites`

Access the raw haplotype *site* data (independent of any particular set of samples): the SNP positions, reference alleles, or alternate alleles that make up the phased site scaffold for a given analysis. This is a lighter-weight way to inspect which sites were phased without also loading genotype calls. Parameters:

- **region**: genome region(s) to query.
- **field**: which site attribute to return — `"POS"` (position), `"REF"` (reference allele) or `"ALT"` (alternate allele).
- **analysis**: which phasing analysis (see `phasing_analysis_ids`); `"default"` uses the configured default.
- **inline_array**, **chunks**: dask performance/chunking knobs.

The example retrieves the positions of phased sites for the `gamb_colu` analysis in the same region as above, then the corresponding reference and alternate alleles.

In [4]:
pos = ag3.haplotype_sites(
    region="3L:15,000,000-16,000,000",
    field="POS",
    analysis="gamb_colu",
)
ref = ag3.haplotype_sites(
    region="3L:15,000,000-16,000,000",
    field="REF",
    analysis="gamb_colu",
)
alt = ag3.haplotype_sites(
    region="3L:15,000,000-16,000,000",
    field="ALT",
    analysis="gamb_colu",
)
pos.shape, pos[:5].compute(), ref[:5].compute(), alt[:5].compute()

((258679,),
 array([15000004, 15000009, 15000010, 15000012, 15000016], dtype=int32),
 array([b'T', b'G', b'A', b'T', b'C'], dtype='|S1'),
 array([b'G', b'T', b'C', b'A', b'A'], dtype='|S1'))